# Property Price & Insights Engine
---

## Import Libraries

In [ ]:
# Import basic librarie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Import advance library for visualization
import seaborn as sns
sns.set_theme(context='paper',style='darkgrid')

# Import warnings
import warnings
warnings.filterwarnings('ignore')

## Load the dataset

In [2]:
df = pd.read_csv(r"C:\Users\Bhawna\OneDrive\Desktop\Real Estate\bengaluru_house_prices.csv")
# Shape of the dataset
print(f"Shape of the dataset:")
print(f"Values: {df.shape[0]}")
print(f"Features: {df.shape[1]}")
# Display top 5 rows
display(df.head())
# Display bottom 5 rows
display(df.tail())

Shape of the dataset:
Values: 13320
Features: 9


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.0
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.0
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.0
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.0
13319,Super built-up Area,Ready To Move,Doddathoguru,1 BHK,NaN,550,1.0,1.0,17.0


## Basic Dataframe Check

In [3]:
check_df = pd.DataFrame({
    "Features": df.columns,
    "Null Values": [df[c].isna().sum() for c in df.columns],
    "Data Type": [df[c].dtype for c in df.columns],
    "Unique Values": [df[c].nunique() for c in df.columns]
})
# Display dataframe
check_df

,Features,Null Values,Data Type,Unique Values
0,area_type,0,object,4
1,availability,0,object,81
2,location,1,object,1305
3,size,16,object,31
4,society,5502,object,2688
5,total_sqft,0,object,2117
6,bath,73,float64,19
7,balcony,609,float64,4
8,price,0,float64,1994


## Data Cleaning

### Removing Unnecessary features

Area Type: Property price is primarily driven by total area and location rather than the technical classification of the land.

Availability: Whether a house is "Ready to Move" or has a future date doesn't significantly impact its core market valuation.

Society: This column has too many missing values and unique names, which creates unnecessary noise for the model.

Balcony: While a feature, the number of balconies is a minor detail compared to major factors like BHK and square footage in determining price.

In [4]:
# First, make a copy 
df2 = df.copy()
# Drop unnecassary features
df2 = df2.drop(['area_type','availability','society','balcony'], axis=1)
# Display df2
df2.head()

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


### Checking null values and treating them

In [5]:
# Sum of null values
display(df2.isnull().sum())
# Removing null values
df2.dropna(inplace=True)
print("Null values removed successfully")
# Checking null values again
display(df2.isna().sum())

location       1
size          16
total_sqft     0
bath          73
price          0
dtype: int64

Null values removed successfully


location      0
size          0
total_sqft    0
bath          0
price         0
dtype: int64

### Feature Engineering

In [ ]:
# Create a new feature from size
df2['bhk'] = df2['size'].apply(lambda x: int(x.split(' ')[0]))
df2.head()

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00,4
2,Uttarahalli,3 BHK,1440,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00,3
4,Kothanur,2 BHK,1200,2.0,51.00,2


### Cleaning total sqft feature

In [25]:
# Values in total sqft
df2['total_sqft'].tail(22)

13298           1015
13299    2830 - 2882
13300           1500
13301           1454
13302           1075
13303            774
13304           1187
13305            500
13306           1200
13307           1805
13308           1527
13309           1675
13310           1050
13311           1500
13312           1262
13313           1345
13314           1715
13315           3453
13316           3600
13317           1141
13318           4689
13319            550
Name: total_sqft, dtype: object

In [26]:
# Defining a function
def clean_sqft_feature(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1]))/2
    try:
        return float(x)
    except:
        return None
    
# Applying function
df2['total_sqft'] = df2['total_sqft'].apply(clean_sqft_feature)
# Display top 5 rows
df2.head()

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3
4,Kothanur,2 BHK,1200.0,2.0,51.00,2
